# Treinamento dos modelos M1 — Dengue

Notebook de execução no Google Colab. Antes de executar, rode [`02_baselines_colab.ipynb`](./02_baselines_colab.ipynb) para gravar no Drive as dobras verificadas e os baselines OOF. O notebook apenas configura o ambiente, valida os arquivos e chama `run_all_models`; o pipeline de modelagem fica no pacote Python.

**Dados:** somente 2022–2023 entram no desenvolvimento. O teste de 2024 permanece congelado e não será aberto por este notebook. **Privacidade:** OOF e arquivos do MLflow contêm dados por caso; mantenha-os no seu Drive e não os compartilhe.

## 1. Preparar o runtime

O projeto exige Python `>=3.13,<3.14`. Se o runtime do Colab não atender a esse intervalo, interrompa e escolha um runtime compatível.

In [ ]:
import sys

if not ((3, 13) <= sys.version_info[:2] < (3, 14)):
    raise RuntimeError(
        f"Python {sys.version_info.major}.{sys.version_info.minor} detectado; "
        "este projeto exige Python >=3.13,<3.14."
    )

print(sys.version)

In [ ]:
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPOSITORY = "https://github.com/PabloThePmX/dengue-comorbidity-risk.git"
BRANCH = "ricardo"
REPO = Path("/content/dengue-comorbidity-risk-ricardo")
remote_branch = subprocess.run(
    ["git", "ls-remote", "--heads", REPOSITORY, f"refs/heads/{BRANCH}"],
    check=True,
    capture_output=True,
    text=True,
)
if not remote_branch.stdout.strip():
    raise RuntimeError(
        f"O branch {BRANCH!r} não está publicado no GitHub. "
        "Publique as alterações ou altere BRANCH para o branch já atualizado."
    )

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPOSITORY, str(REPO)],
        check=True,
    )
else:
    current_branch = subprocess.run(
        ["git", "-C", str(REPO), "branch", "--show-current"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if current_branch != BRANCH:
        raise RuntimeError(
            f"O clone existente está em {current_branch!r}, não em {BRANCH!r}. "
            "Reinicie o runtime do Colab para clonar o branch correto."
        )
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )

required_files = [
    REPO / "pyproject.toml",
    REPO / "config" / "base.yaml",
    REPO / "config" / "esperado.yaml",
    REPO / "src" / "dengue_sinan" / "runner.py",
    REPO / "src" / "dengue_sinan" / "folds.py",
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "O branch clonado não contém os arquivos exigidos pelo treinamento M1: "
        + ", ".join(missing_files)
    )

%cd /content/dengue-comorbidity-risk-ricardo
%pip install -e .

## 2. Configurar o Drive e verificar os pré-requisitos

Os Parquets e DBCs devem ter sido preparados pelo notebook de profiling e estar no Drive. O passo 02 também precisa ter gerado `m1/dobras.npz`, `reports/verificacao/T-2.3.json` e os três arquivos OOF dos baselines.

In [ ]:
import json

from dengue_sinan.config import load_config

PROJECT_DIR = Path("/content/drive/MyDrive/dengue-comorbidity-risk")
DATA_DIR = PROJECT_DIR / "data"
REPORTS_DIR = PROJECT_DIR / "reports"
MLRUNS_DIR = PROJECT_DIR / "mlruns"
MODELS_DIR = PROJECT_DIR / "models"
M1_DIR = PROJECT_DIR / "m1"
for directory in (DATA_DIR, REPORTS_DIR, MLRUNS_DIR, MODELS_DIR, M1_DIR):
    directory.mkdir(parents=True, exist_ok=True)

config = load_config(REPO / "config" / "base.yaml")
config["caminhos"].update(
    {
        "dados": str(DATA_DIR),
        "relatorios": str(REPORTS_DIR),
        "mlruns": str(MLRUNS_DIR),
        "modelos": str(MODELS_DIR),
        "m1": str(M1_DIR),
        "dobras": str(M1_DIR / "dobras.npz"),
        "esperado": str(REPO / "config" / "esperado.yaml"),
    }
)

manifest_path = DATA_DIR / "manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError(
        f"Manifesto ausente: {manifest_path}. Execute primeiro o notebook de profiling."
    )
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
required_data_files = []
for year in config["anos_dev"]:
    entry = manifest.get(str(year))
    if not isinstance(entry, dict) or not {"parquet", "arquivo"}.issubset(entry):
        raise ValueError(f"Manifesto sem metadados completos para {year}.")
    required_data_files.extend(
        [DATA_DIR / entry["parquet"], DATA_DIR / "raw" / entry["arquivo"]]
    )
missing_data_files = [str(path) for path in required_data_files if not path.is_file()]
if missing_data_files:
    raise FileNotFoundError(
        "Arquivos de desenvolvimento ausentes no Drive: " + ", ".join(missing_data_files)
    )

prerequisites = [
    M1_DIR / "dobras.npz",
    REPORTS_DIR / "verificacao" / "T-2.3.json",
    MLRUNS_DIR / "oof" / "b0.parquet",
    MLRUNS_DIR / "oof" / "b1.parquet",
    MLRUNS_DIR / "oof" / "b2.parquet",
]
missing_prerequisites = [str(path) for path in prerequisites if not path.is_file()]
if missing_prerequisites:
    raise FileNotFoundError(
        "Pré-requisitos ausentes. Execute primeiro 02_baselines_colab.ipynb. Ausentes: "
        + ", ".join(missing_prerequisites)
    )

print(f"Desenvolvimento: anos {config['anos_dev']} (sem abrir 2024)")
print(f"Dobras: {M1_DIR / 'dobras.npz'}")
print(f"Resultados persistentes: {PROJECT_DIR}")

## 3. Treinar ou retomar M1

O runner retoma modelos com `CONCLUIDO.json` válido, executa a seleção/calibração, SHAP global e o retreino final. O arquivo `sessao-m1.json` registra progresso e verificações. A execução real pode levar horas; mantenha o runtime conectado.

In [ ]:
from dengue_sinan.runner import run_all_models

resultado = run_all_models(config)
print(f"Modelo selecionado: {resultado['modelo']}")
print(f"Limiar congelado: {resultado['limiar']:.6f}")
print(f"Artefato final no Drive: {resultado['artefato']}")
print(f"Relatório da sessão: {resultado['sessao']}")

In [ ]:
from google.colab import files

session_report = REPORTS_DIR / "verificacao" / "sessao-m1.json"
if not session_report.is_file():
    raise FileNotFoundError(f"Relatório da sessão não encontrado: {session_report}")
files.download(str(session_report))